# PyTorch Debugging and Performance

**PyTorch Interview Preparation · Notebook 09 · Estimated study time: 2.5–3.5 hours**

This notebook treats debugging as evidence gathering. You will repeatedly see a symptom, form a hypothesis, inspect the pipeline, fix one thing, and verify the result.

> **Core rule:** Do not randomly change hyperparameters until the bug disappears. Inspect the pipeline systematically.


## 1. Setup 🟢 Easy

Only PyTorch and the Python standard library are used. All runnable examples work on CPU; CUDA-specific paths are guarded and intentionally small.


In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
torch.set_num_threads(max(1, min(torch.get_num_threads(), 4)))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. A Systematic Debugging Workflow 🔥 Interview Essential

```text
1. Reproduce the problem
        ↓
2. Check shapes → dtypes → devices
        ↓
3. Check values, targets, and loss
        ↓
4. Check gradients and parameter updates
        ↓
5. Check train/eval behavior
        ↓
6. Check memory and performance
```

Change one thing, rerun the smallest reproducible case, and record whether the evidence supports your hypothesis. Start with one batch; then try to overfit 8–32 examples.


## 3. Inspecting Tensor State 🟢 Easy

### 🔥 Interview Essential — shape, dtype, device

A large fraction of PyTorch runtime failures reduce to incompatible shapes, dtypes, or devices. Ask: What came in? What does the next operation expect? Where is each tensor? Inspect values as well as metadata.


In [ ]:
def inspect_tensor(name, x):
    print(name)
    print("  shape:", tuple(x.shape))
    print("  dtype:", x.dtype)
    print("  device:", x.device)
    print("  requires_grad:", x.requires_grad)
    if x.numel() > 0 and x.is_floating_point():
        finite = torch.isfinite(x)
        print("  finite:", finite.all().item())
        if finite.any():
            values = x[finite]
            print("  min/max/mean:", values.min().item(), values.max().item(), values.mean().item())


def inspect_model(model):
    total = trainable = 0
    for name, param in model.named_parameters():
        total += param.numel()
        trainable += param.numel() if param.requires_grad else 0
        print(name, tuple(param.shape), param.dtype, param.device, param.requires_grad)
    print("Total parameters:", total)
    print("Trainable parameters:", trainable)


sample = torch.randn(4, 3, requires_grad=True)
inspect_tensor("sample", sample)


## 4. Shape Mismatches 🟢 Easy

### Linear layer

```python
model = nn.Linear(10, 3)
X = torch.randn(32, 8)  # broken: last dimension is 8
logits = model(X)
```

**Question:** What is wrong? The layer expects 10 input features. Fix the data to `(32, 10)` or redesign the layer to accept 8—based on the data contract, not merely to silence the error.


In [ ]:
linear = nn.Linear(10, 3)
broken_X = torch.randn(32, 8)
try:
    linear(broken_X)
except RuntimeError as error:
    print("Expected shape error:", str(error).split("\n")[0])

fixed_X = torch.randn(32, 10)
assert linear(fixed_X).shape == (32, 3)


### Cross-entropy, BCE, and CNN shapes

| Task | Output | Typical target |
|---|---:|---:|
| single-label multiclass | `(batch, classes)` logits | `(batch,)` integer indices |
| binary with one logit | `(batch, 1)` logits | `(batch, 1)` floats |
| multilabel | `(batch, classes)` logits | `(batch, classes)` float multi-hot |

For class-index targets, `(32, 1)` can become `(32,)` with `squeeze(1)`. Do not blindly call `squeeze()` on unknown dimensions. `Conv2d` normally expects **NCHW**, not NHWC.


In [ ]:
logits_ce = torch.randn(32, 10)
targets_ce = torch.randint(0, 10, (32, 1)).squeeze(1)
assert nn.CrossEntropyLoss()(logits_ce, targets_ce).ndim == 0

logits_bce = torch.randn(32, 1)
targets_bce = torch.randint(0, 2, (32,)).float().unsqueeze(1)
assert logits_bce.shape == targets_bce.shape
assert nn.BCEWithLogitsLoss()(logits_bce, targets_bce).ndim == 0

conv = nn.Conv2d(3, 16, kernel_size=3)
images_nhwc = torch.randn(8, 32, 32, 3)
images_nchw = images_nhwc.permute(0, 3, 1, 2)
assert conv(images_nchw).shape == (8, 16, 30, 30)
print("NCHW:", images_nchw.shape)


**Shape exercises**

1. A linear layer is `Linear(64, 8)` and input is `(16, 32)`. Name both valid categories of fix.
2. CE logits are `(24, 5)` and targets `(24, 1)`. Fix only the known singleton dimension.
3. BCE logits are `(24,)` and targets `(24, 1)`. Choose and document one consistent interface.
4. A CNN receives `(8, 28, 28)`. Which dimension is missing?
5. An image batch is `(8, 28, 28, 1)`. Write the `permute` call.
6. Why could indiscriminate `squeeze()` fail when batch size equals one?


## 5. Dtype Mismatches 🟢 Easy

### 🔥 Interview Essential

Embedding indices and ordinary class-index targets use integer index dtypes such as `torch.long`. Most model inputs match the parameter floating dtype. Inspect with `next(model.parameters()).dtype`.


In [ ]:
embedding = nn.Embedding(1000, 64)
tokens_float = torch.tensor([[1.0, 2.0, 3.0]])
try:
    embedding(tokens_float)
except RuntimeError as error:
    print("Expected embedding dtype error:", str(error).split("\n")[0])

tokens = tokens_float.long()
assert embedding(tokens).shape == (1, 3, 64)

model_fp32 = nn.Linear(4, 2)
X_fp64 = torch.randn(3, 4, dtype=torch.float64)
print("input/model dtype:", X_fp64.dtype, next(model_fp32.parameters()).dtype)
X_fp32 = X_fp64.float()
assert model_fp32(X_fp32).dtype == torch.float32


**Debugging exercise:** `CrossEntropyLoss` receives logits `(16, 5)` and targets created as `[0., 1., 4., ...]`. Inspect target meaning, shape, range, and dtype; for ordinary class indices convert to `long`. Float targets can be valid in other CE formulations, so do not cast without understanding the target contract.


## 6. Device Mismatches 🟢 Easy

### 🔥 Interview Essential

The fastest diagnostic is:

```python
print(next(model.parameters()).device)
print(X.device)
print(y.device)
```

Move the model once and move each batch consistently. Avoid hard-coding `.cuda()` when code must also support CPU or MPS.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_model = nn.Linear(6, 3).to(device)
X_device = torch.randn(4, 6).to(device)
y_device = torch.randint(0, 3, (4,)).to(device)
print("model/X/y:", next(device_model.parameters()).device, X_device.device, y_device.device)
assert device_model(X_device).device == device


## 7. Target and Loss Mismatches 🔥 Interview Essential

| Problem | Usual output/target | Usual loss |
|---|---|---|
| single-label, 10 classes | 10 raw logits + one class index | `CrossEntropyLoss` |
| multilabel, 10 labels | 10 raw logits + 10 float indicators | `BCEWithLogitsLoss` |
| binary, one output | one raw logit + matching float target | `BCEWithLogitsLoss` |

### 🔥 Very Common Interview Bug

Pass **raw logits** to `CrossEntropyLoss`; do not apply softmax first. Pass raw logits to `BCEWithLogitsLoss`; do not apply sigmoid first. These losses incorporate the activation in a numerically stable formulation.


In [ ]:
logits = torch.randn(12, 4, requires_grad=True)
y = torch.randint(0, 4, (12,), dtype=torch.long)
loss = nn.CrossEntropyLoss()(logits, y)
assert torch.isfinite(loss)

values, counts = torch.unique(y, return_counts=True)
print("labels/counts:", list(zip(values.tolist(), counts.tolist())))
print("target range:", y.min().item(), y.max().item())
assert 0 <= y.min() and y.max() < logits.shape[1]


## 8. Debugging Loss That Does Not Decrease 🔥 Interview Essential

1. Check data and labels: finiteness, ranges, distributions, alignment.
2. Check output/target shapes, target format, and loss choice.
3. Check activation/loss pairing.
4. Check the learning rate; too high may oscillate/diverge, too low may move imperceptibly.
5. Verify gradients exist and are finite.
6. Verify `optimizer.step()` changes parameters.
7. Check frozen parameters and optimizer ownership.
8. Check `train()`/`eval()` modes.
9. Try to overfit 8–32 examples.
10. Inspect normalization and preprocessing.

High train performance with worsening validation is likely overfitting. Flat train loss with unchanged parameters points to a mechanical training problem. High train and validation loss can be underfitting, poor optimization, weak features, or a bug—verify mechanics first.


### Tiny-batch overfitting test 🔥 Excellent Practical Debugging Technique

If a model cannot memorize a tiny, clean dataset, suspect the loop, targets, gradient flow, learning rate, or capacity. Passing this test does not prove generalization; it proves basic trainability.


In [ ]:
torch.manual_seed(42)
tiny_X = torch.randn(24, 6)
tiny_y = ((tiny_X[:, 0] + 0.7 * tiny_X[:, 1]) > 0).long()
tiny_model = nn.Sequential(nn.Linear(6, 24), nn.ReLU(), nn.Linear(24, 2))
tiny_opt = optim.Adam(tiny_model.parameters(), lr=0.03)
tiny_loss_fn = nn.CrossEntropyLoss()

for step in range(160):
    tiny_opt.zero_grad()
    tiny_logits = tiny_model(tiny_X)
    tiny_loss = tiny_loss_fn(tiny_logits, tiny_y)
    tiny_loss.backward()
    tiny_opt.step()

with torch.inference_mode():
    tiny_accuracy = (tiny_model(tiny_X).argmax(1) == tiny_y).float().mean().item()
print("tiny-set loss/accuracy:", tiny_loss.item(), tiny_accuracy)
assert tiny_accuracy >= 0.95


**Broken-loop exercise:** A loop calls `zero_grad()`, forward, loss, and `step()` but omits `backward()`. Predict the evidence: gradients remain `None`, parameters do not change, and loss cannot improve through learning. Repair it and validate all three claims.


## 9. Checking Parameter Updates 🟡 Medium

Clone parameters before one step and compare afterward. If none change, inspect gradients, optimizer ownership, `requires_grad`, step ordering, and update magnitude.


In [ ]:
check_model = nn.Linear(5, 2)
check_opt = optim.SGD(check_model.parameters(), lr=0.1)
check_X = torch.randn(8, 5)
check_y = torch.randint(0, 2, (8,))
before = {name: p.detach().clone() for name, p in check_model.named_parameters()}

check_opt.zero_grad()
check_loss = nn.CrossEntropyLoss()(check_model(check_X), check_y)
check_loss.backward()
check_opt.step()

changed = {name: not torch.equal(before[name], p.detach()) for name, p in check_model.named_parameters()}
print(changed)
assert all(changed.values())


Two realistic ownership bugs:

```python
optimizer = Adam(model_a.parameters())
loss = loss_fn(model_b(X), y)  # model_b ran, model_a is stepped
```

and parameters left frozen with `requires_grad=False`. `inspect_model()` exposes trainability, while checking optimizer parameter identities exposes ownership.


## 10. Checking Gradients 🔥 Interview Essential

`grad is None` means no gradient was populated for that parameter in the latest backward path. That can be intentional (frozen/unused branch) or a bug (`detach`, wrong model, no backward). A near-zero norm suggests weak flow or an inactive path; an extremely large norm suggests instability. There is no universal good threshold.


In [ ]:
def gradient_report(model):
    for name, param in model.named_parameters():
        if not param.requires_grad:
            print(name, "FROZEN")
        elif param.grad is None:
            print(name, "NO GRAD")
        else:
            print(name, "norm=", param.grad.norm().item(), "finite=", torch.isfinite(param.grad).all().item())


def global_grad_norm(model):
    norms = [p.grad.detach().norm() for p in model.parameters() if p.grad is not None]
    return torch.stack(norms).norm().item() if norms else 0.0


def global_parameter_norm(model):
    norms = [p.detach().norm() for p in model.parameters()]
    return torch.stack(norms).norm().item() if norms else 0.0


gradient_report(check_model)
print("global grad norm:", global_grad_norm(check_model))
print("parameter norm:", global_parameter_norm(check_model))


## 11. Vanishing Gradients 🟡 Medium

Vanishing gradients are extremely small gradients in earlier layers, making updates very slow. Deep chains, repeated multiplicative effects, saturating sigmoid/tanh activations, and poor initialization can contribute. Inspect per-layer norms across depth and time; do not diagnose from one number alone.


In [ ]:
torch.manual_seed(7)
vanish_model = nn.Sequential(*sum(([nn.Linear(8, 8), nn.Sigmoid()] for _ in range(6)), []), nn.Linear(8, 1))
vanish_out = vanish_model(torch.randn(16, 8)).mean()
vanish_out.backward()
vanish_norms = [(name, p.grad.norm().item()) for name, p in vanish_model.named_parameters() if p.grad is not None]
print("first/last parameter grad norms:", vanish_norms[0], vanish_norms[-1])
assert all(torch.isfinite(p.grad).all() for p in vanish_model.parameters())


## 12. Exploding Gradients and Clipping 🔥 Interview Essential

Symptoms include rapidly growing gradient norms, unstable parameter updates, huge loss, and eventually NaN/Inf. Investigate the data, learning rate, initialization, normalization, and architecture. Clipping can control the update but may only mask a deeper cause.


In [ ]:
clip_model = nn.Linear(4, 1)
large_X = torch.randn(16, 4) * 1_000
clip_model(large_X).pow(2).mean().backward()
before_clip = global_grad_norm(clip_model)
returned_norm = torch.nn.utils.clip_grad_norm_(clip_model.parameters(), max_norm=1.0)
after_clip = global_grad_norm(clip_model)
print("before/returned/after:", before_clip, returned_norm.item(), after_clip)
assert after_clip <= 1.0001


## 13. NaN and Inf Debugging 🔥 Interview Essential

Common sources: bad inputs, divide-by-zero, invalid log/sqrt, overflow, excessive learning rate, exploding gradients, unstable reduced precision, invalid normalization, or invalid targets. Stop at the first non-finite boundary; silently continuing destroys evidence.


In [ ]:
nonfinite = torch.tensor([1.0, float("nan"), float("inf"), -float("inf")])
print("isnan:   ", torch.isnan(nonfinite))
print("isinf:   ", torch.isinf(nonfinite))
print("isfinite:", torch.isfinite(nonfinite))

unsafe_normalized = torch.tensor([1.0, 2.0]) / torch.tensor(0.0)
invalid_logs = torch.log(torch.tensor([1.0, 0.0, -1.0]))
print("division finite:", torch.isfinite(unsafe_normalized))
print("logs:", invalid_logs)
safe_std = torch.tensor(0.0) + 1e-8  # only appropriate if this matches the intended mathematics
assert torch.isfinite(torch.tensor([1.0, 2.0]) / safe_std).all()


In [ ]:
def assert_finite_tensor(name, x):
    assert torch.isfinite(x).all(), f"{name} contains non-finite values"


def assert_gradients_finite(model):
    for name, param in model.named_parameters():
        if param.grad is not None:
            assert torch.isfinite(param.grad).all(), f"Non-finite gradient: {name}"


def assert_parameters_finite(model):
    for name, param in model.named_parameters():
        assert torch.isfinite(param).all(), f"Non-finite parameter: {name}"


data_X = torch.randn(100, 5)
data_y = torch.randint(0, 3, (100,))
print("X stats:", data_X.mean().item(), data_X.std().item(), data_X.min().item(), data_X.max().item())
print("X/y finite:", torch.isfinite(data_X).all().item(), torch.isfinite(data_y.float()).all().item())
assert_finite_tensor("data_X", data_X)


## 14. Anomaly Detection 🟡 Medium

`torch.autograd.set_detect_anomaly(True)` helps trace the backward operation that generated invalid gradients. It is expensive, so scope it to debugging—not routine training.


In [ ]:
anomaly_x = torch.tensor([0.0], requires_grad=True)
try:
    with torch.autograd.detect_anomaly():
        anomaly_loss = torch.sqrt(anomaly_x).sum()
        anomaly_loss.backward()
except RuntimeError as error:
    print("Anomaly detection caught:", str(error).split("\n")[0])
finally:
    torch.autograd.set_detect_anomaly(False)


## 15. In-Place Operation Errors 🟡 Medium

Methods ending in `_`, such as `add_()` and `relu_()`, mutate storage. Autograd may need the original value for backward, producing “a variable needed for gradient computation has been modified by an inplace operation.” Search for underscore operations **and indexed assignments/overwrites**. Prefer an out-of-place equivalent while isolating the bug.


In [ ]:
inplace_x = torch.tensor([2.0], requires_grad=True)
inplace_y = inplace_x * inplace_x
with torch.no_grad():
    inplace_x.add_(1.0)
try:
    inplace_y.backward()
except RuntimeError as error:
    print("Expected in-place Autograd error:", str(error).split("\n")[0])


## 16. `train()` / `eval()` Bugs 🔥 Interview Essential

These methods change module behavior; they do **not** enable/disable gradients. Dropout is stochastic in training and disabled in evaluation. BatchNorm updates/uses batch statistics in training and uses running statistics in evaluation. Gradient context is controlled separately by `no_grad()` or `inference_mode()`.


In [ ]:
mode_model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Dropout(0.5), nn.Linear(20, 2))
mode_X = torch.ones(4, 10)
mode_model.train()
train_a, train_b = mode_model(mode_X), mode_model(mode_X)
mode_model.eval()
with torch.inference_mode():
    eval_a, eval_b = mode_model(mode_X), mode_model(mode_X)
print("train equal/eval equal:", torch.equal(train_a, train_b), torch.equal(eval_a, eval_b))
assert torch.equal(eval_a, eval_b)


## 17. Dropout and BatchNorm Debugging 🟡 Medium

Call `model.train()` at the start of every training phase and `model.eval()` before validation. Tiny batches make BatchNorm batch estimates noisy; a wrong channel/feature dimension can be a separate shape bug. If validation remains random, inspect mode first. If validation builds graphs, inspect gradient context separately.


## 18. Memory Leaks and Graph Retention 🔥 Interview Essential

```python
losses.append(loss)                    # can retain the graph
predictions_saved.append(predictions) # can retain graph and GPU storage
```

Use `loss.item()` for scalar logging, or `predictions.detach().cpu()` for later metrics. `retain_graph=True` deliberately keeps graph state and should not be used casually to suppress repeated-backward errors. Python references—not only the allocator—determine whether graph tensors remain alive.


In [ ]:
memory_model = nn.Linear(3, 2)
memory_X = torch.randn(5, 3)
memory_y = torch.randint(0, 2, (5,))
memory_logits = memory_model(memory_X)
memory_loss = nn.CrossEntropyLoss()(memory_logits, memory_y)
safe_losses = [memory_loss.item()]
safe_predictions = [memory_logits.detach().cpu()]
assert isinstance(safe_losses[0], float) and safe_predictions[0].grad_fn is None


## 19. CUDA OOM Debugging 🔥 Interview Essential

1. Confirm batch, sequence, and image sizes.
2. Look for stored graph-connected losses/outputs and `retain_graph=True`.
3. Check whether multiple models or batches remain referenced.
4. Inspect allocated, reserved, and peak allocated memory.
5. Reduce batch size; consider mixed precision or gradient accumulation when appropriate.
6. Profile before complex changes.

Reserved memory includes allocator-managed cached blocks; allocated memory is live tensor memory. `empty_cache()` may release unused cached blocks to other processes, but it does not free live tensors or repair a leak.


In [ ]:
def cuda_memory_report():
    if not torch.cuda.is_available():
        print("CUDA unavailable")
        return
    print("allocated MB:", torch.cuda.memory_allocated() / 1024**2)
    print("reserved MB:", torch.cuda.memory_reserved() / 1024**2)
    print("peak allocated MB:", torch.cuda.max_memory_allocated() / 1024**2)


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
cuda_memory_report()


## 20. DataLoader Bottlenecks 🔥 Interview Essential for ML Engineering

A GPU repeatedly waiting between batches suggests an input bottleneck, but a trace is evidence—not a low utilization number alone. Possible causes include slow storage, costly Python transforms, too few workers, a saturated CPU, small batches, and no transfer/compute overlap. Benchmark worker counts; more workers are not always faster, especially in notebooks or on macOS/Windows.


In [ ]:
loader_dataset = TensorDataset(torch.randn(1024, 20), torch.randint(0, 4, (1024,)))
loader = DataLoader(loader_dataset, batch_size=64, shuffle=False, num_workers=0)
start = time.perf_counter()
seen = 0
for batch_X, batch_y in loader:
    seen += len(batch_X)
load_elapsed = time.perf_counter() - start
print(f"iterated {seen} in-memory samples in {load_elapsed:.4f}s")
assert seen == len(loader_dataset)


## 21. Low GPU Utilization 🔥 Interview Essential

| Possible cause | Possible sign |
|---|---|
| DataLoader slow | gaps before batches |
| batch/model too small | launch overhead dominates |
| CPU work in loop | GPU waits |
| frequent `.item()` or prints | synchronization/host overhead |
| repeated transfers | copy-bound path |
| Python element loops | many tiny operations |

Separate data iteration from compute on already-created tensors. Then use a profiler to confirm. Avoid calling `model.to(device)` inside the batch loop, avoid CPU↔GPU ping-pong, and log at a deliberate cadence.


## 22. Timing Code Correctly 🟡 Medium

Use warmup and multiple repetitions. CUDA launches are asynchronous, so synchronize before starting and after the measured operation. Synchronization itself affects the program; place it at measurement boundaries, not throughout normal training.


In [ ]:
timing_model = nn.Linear(128, 64).to(device)
timing_X = torch.randn(256, 128, device=device)
for _ in range(5):
    _ = timing_model(timing_X)
if device.type == "cuda":
    torch.cuda.synchronize()

times = []
with torch.inference_mode():
    for _ in range(20):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = timing_model(timing_X)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
print("mean forward seconds:", sum(times) / len(times))


In [ ]:
import torch.utils.benchmark as benchmark

bench_x = torch.randn(64, 64)
bench_y = torch.randn(64, 64)
timer = benchmark.Timer(stmt="x @ y", globals={"x": bench_x, "y": bench_y})
print(timer.timeit(20))


## 23. Basic PyTorch Profiling 🟡 Medium

`torch.profiler` identifies expensive operators, many tiny calls, unexpected CPU work, repeated copies/transfers, and memory-heavy operations. This minimal example profiles CPU work; add `ProfilerActivity.CUDA` only when CUDA is available.


In [ ]:
from torch.profiler import profile, ProfilerActivity

profile_model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 8))
profile_X = torch.randn(128, 32)
with profile(activities=[ProfilerActivity.CPU]) as prof:
    _ = profile_model(profile_X)
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=6))


## 24. Vectorization and Python Loops 🔥 Interview Essential

Vectorized tensor operations reduce Python overhead and use optimized kernels. They are especially important on accelerators, where tiny loop-launched kernels underutilize the device.


In [ ]:
vec_x = torch.arange(10.0)
loop_result = torch.empty_like(vec_x)
for i in range(len(vec_x)):
    loop_result[i] = vec_x[i] * 2
vector_result = vec_x * 2
assert torch.equal(loop_result, vector_result)


### Eight vectorization exercises

Rewrite each without a Python element loop.

1. Double every value in `x`. → `x * 2`
2. Square every value. → `x.square()`
3. Sum rows of a matrix. → `x.sum(dim=1)`
4. Count positive elements per row. → `(x > 0).sum(dim=1)`
5. Normalize each row by its norm. → `x / x.norm(dim=1, keepdim=True).clamp_min(eps)`
6. Select rows whose labels equal 2. → `x[y == 2]`
7. Pairwise dot products of rows in `x` and `y`. → `x @ y.T`
8. Replace negative values with zero. → `x.clamp_min(0)` or `torch.relu(x)`

For each: test equality against a small loop implementation, then benchmark both.


## 25. Efficient Metric Logging 🟡 Medium

Calling `.item()` and printing every accelerator step can synchronize and add host overhead. Accumulate detached values and report periodically. For inference outputs, batch work under `model.eval()` and `torch.inference_mode()`, then detach/copy only the results you need.


In [ ]:
metric_total = torch.zeros(())
for step in range(20):
    metric_total += torch.tensor(float(step))
if step % 10 != 0:
    print("mean metric:", (metric_total / 20).item())


## 26. Common Performance Anti-Patterns 🟡 Medium

- Repeated `torch.cat` in a loop reallocates/copies; append chunks and concatenate once.
- Creating constant tensors in a hot loop repeats setup/allocation; create once outside.
- Moving the model every batch repeats expensive transfers; move it once.
- CPU→GPU→CPU round trips make the device wait.
- Per-sample inference wastes batching opportunities.
- Frequent scalar extraction/printing creates synchronization and logging overhead.


In [ ]:
batches = [torch.randn(4, 10) for _ in range(5)]
chunks = []
for batch in batches:
    chunks.append(batch)
combined = torch.cat(chunks, dim=0)
assert combined.shape == (20, 10)


### Learning-rate diagnosis 🔥 Interview Essential

A too-high rate can oscillate, enlarge parameters/gradients, diverge, or become non-finite. A too-low rate can produce finite gradients and tiny parameter changes while loss improves extremely slowly. Neither symptom proves the rate is the only cause. Compare controlled runs, track loss/gradient/parameter norms, and verify data and mechanics first.


## 27. Debugging Scenarios

For every scenario: read the symptom and code, state a hypothesis, name the evidence you would collect, then reveal the answer. These are interview-style diagnosis drills—not API trivia.


### Scenario 1: Loss is constant 🟢 Easy

**Symptoms:** Loss is identical each epoch; grads are absent.

**Code:**

```python
optimizer.zero_grad(); logits=model(X); loss=loss_fn(logits,y); optimizer.step()
```

**Question:** Which training operation is missing?

**Hint:** Inspect `param.grad` immediately before `step()`.

<details><summary>Answer and why</summary>

**Answer:** Call `loss.backward()` before `optimizer.step()`.

**Why:** The optimizer needs populated gradients.

</details>


### Scenario 2: No parameter update 🟡 Medium

**Symptoms:** Backward runs, but no weights change.

**Code:**

```python
loss.backward(); optimizer.zero_grad(); optimizer.step()
```

**Question:** Why is the update empty?

**Hint:** Follow the gradient lifecycle in order.

<details><summary>Answer and why</summary>

**Answer:** Zeroing after backward erases gradients before the step.

**Why:** Use zero_grad → forward → loss → backward → step.

</details>


### Scenario 3: Gradients grow every batch 🟡 Medium

**Symptoms:** Norms increase even on repeated identical batches.

**Code:**

```python
loss.backward(); optimizer.step()
```

**Question:** What likely accumulates?

**Hint:** Look for the start-of-step reset.

<details><summary>Answer and why</summary>

**Answer:** `optimizer.zero_grad()` is missing.

**Why:** PyTorch accumulates gradients by default.

</details>


### Scenario 4: Validation changes the model 🟡 Medium

**Symptoms:** Validation metrics affect later training.

**Code:**

```python
for X,y in val_loader: loss.backward(); optimizer.step()
```

**Question:** What must be removed?

**Hint:** Evaluation should not optimize.

<details><summary>Answer and why</summary>

**Answer:** Remove backward/step and use eval plus inference mode.

**Why:** Validation must observe rather than mutate parameters.

</details>


### Scenario 5: Validation is random 🟡 Medium

**Symptoms:** Same validation input produces changing predictions.

**Code:**

```python
model.train(); output=model(val_X)
```

**Question:** What mode is likely wrong?

**Hint:** The model contains Dropout.

<details><summary>Answer and why</summary>

**Answer:** Call `model.eval()` before validation.

**Why:** Dropout remains stochastic in train mode.

</details>


### Scenario 6: Memory grows each epoch 🟡 Medium

**Symptoms:** Accelerator memory rises with output history.

**Code:**

```python
epoch_outputs.append(output)
```

**Question:** What does the list retain?

**Hint:** Inspect `output.grad_fn` and device.

<details><summary>Answer and why</summary>

**Answer:** Store `output.detach().cpu()` or only needed metrics.

**Why:** The list retains graph-connected device tensors.

</details>


### Scenario 7: OOM only in validation 🟡 Medium

**Symptoms:** Training fits; validation runs out of memory.

**Code:**

```python
model.eval(); outputs.append(model(X))
```

**Question:** Name two likely fixes.

**Hint:** Mode and gradient tracking are separate.

<details><summary>Answer and why</summary>

**Answer:** Use inference mode and avoid saving graph/device outputs.

**Why:** Eval mode alone does not disable Autograd.

</details>


### Scenario 8: Low GPU utilization 🟡 Medium

**Symptoms:** Long gaps occur before each device batch.

**Code:**

```python
batch = expensive_python_transform(batch); batch=batch.to(device)
```

**Question:** Which subsystem is suspect?

**Hint:** Time data preparation separately.

<details><summary>Answer and why</summary>

**Answer:** The input/CPU pipeline is a leading hypothesis.

**Why:** The GPU waits while Python preprocessing runs.

</details>


### Scenario 9: NaN after ten steps 🟡 Medium

**Symptoms:** Grad norms grow; LR is 10.

**Code:**

```python
optimizer = SGD(model.parameters(), lr=10)
```

**Question:** What should you inspect first?

**Hint:** Find the first non-finite value.

<details><summary>Answer and why</summary>

**Answer:** Check inputs/loss/grads; compare a lower LR and consider clipping after diagnosis.

**Why:** The high LR is suspicious, not proof by itself.

</details>


### Scenario 10: Backbone grad is None 🟡 Medium

**Symptoms:** Classifier learns, backbone has no gradients.

**Code:**

```python
features = backbone(X).detach(); logits=classifier(features)
```

**Question:** Where is flow broken?

**Hint:** Inspect tensor `requires_grad` and `grad_fn`.

<details><summary>Answer and why</summary>

**Answer:** `.detach()` severs the graph to the backbone.

**Why:** Only downstream classifier parameters receive gradients.

</details>


### Scenario 11: Frozen backbone 🟡 Medium

**Symptoms:** Fine-tuning updates only the head.

**Code:**

```python
for p in backbone.parameters(): p.requires_grad=False
```

**Question:** What state contradicts the goal?

**Hint:** Use `inspect_model()`.

<details><summary>Answer and why</summary>

**Answer:** Re-enable selected backbone parameters and ensure optimizer owns them.

**Why:** Frozen parameters do not receive gradients.

</details>


### Scenario 12: Slow inference 🟡 Medium

**Symptoms:** One sample is evaluated at a time.

**Code:**

```python
for sample in dataset: model(sample.unsqueeze(0))
```

**Question:** How can throughput improve?

**Hint:** Look at batching and gradient context.

<details><summary>Answer and why</summary>

**Answer:** Batch samples, use eval/inference_mode, and avoid repeated transfers.

**Why:** Per-sample overhead dominates.

</details>


### Scenario 13: Embedding dtype error 🟢 Easy

**Symptoms:** Embedding rejects token tensor.

**Code:**

```python
tokens=torch.tensor([[1.,2.,3.]]); embedding(tokens)
```

**Question:** What dtype is expected?

**Hint:** Indices are not continuous features.

<details><summary>Answer and why</summary>

**Answer:** Use integer indices, normally `tokens.long()`.

**Why:** Embedding performs indexed lookup.

</details>


### Scenario 14: CE target shape error 🟢 Easy

**Symptoms:** Logits are `(32,10)`, target `(32,1)`.

**Code:**

```python
loss=CrossEntropyLoss()(logits,target)
```

**Question:** What target shape is usual?

**Hint:** One class index per sample.

<details><summary>Answer and why</summary>

**Answer:** Use `(32,)`, e.g. `squeeze(1)` when that dimension is known singleton.

**Why:** Class-index CE does not expect a trailing singleton.

</details>


### Scenario 15: BCE size mismatch 🟢 Easy

**Symptoms:** Logits `(32,1)`, targets `(32,)`.

**Code:**

```python
BCEWithLogitsLoss()(logits, targets)
```

**Question:** How should the contract change?

**Hint:** BCE expects matching shapes.

<details><summary>Answer and why</summary>

**Answer:** Unsqueeze targets or consistently emit `(batch,)` logits.

**Why:** Elementwise binary loss aligns positions.

</details>


### Scenario 16: CNN channel error 🟢 Easy

**Symptoms:** Conv2d sees 224 channels.

**Code:**

```python
X=torch.randn(8,224,224,3); conv(X)
```

**Question:** Which layout is wrong?

**Hint:** Conv2d expects NCHW.

<details><summary>Answer and why</summary>

**Answer:** Use `X.permute(0,3,1,2)`.

**Why:** The last channel dimension must move to index 1.

</details>


### Scenario 17: Wrong optimizer model 🟡 Medium

**Symptoms:** Forward model gradients exist but weights do not change.

**Code:**

```python
opt=Adam(model_a.parameters()); loss_fn(model_b(X),y).backward(); opt.step()
```

**Question:** Which object is stepped?

**Hint:** Compare optimizer parameter identities.

<details><summary>Answer and why</summary>

**Answer:** Attach the optimizer to `model_b`.

**Why:** The forward and optimizer reference different models.

</details>


### Scenario 18: Softmax before CE 🟡 Medium

**Symptoms:** Training is unnecessarily weak/unstable.

**Code:**

```python
loss=CrossEntropyLoss()(softmax(logits,1), y)
```

**Question:** What should enter the loss?

**Hint:** Read the loss input contract.

<details><summary>Answer and why</summary>

**Answer:** Pass raw logits.

**Why:** CE internally applies a stable log-softmax formulation.

</details>


### Scenario 19: Sigmoid before BCE logits 🟡 Medium

**Symptoms:** Binary training behaves poorly.

**Code:**

```python
loss=BCEWithLogitsLoss()(sigmoid(logits), y)
```

**Question:** What is duplicated?

**Hint:** The loss name is a clue.

<details><summary>Answer and why</summary>

**Answer:** Pass raw logits.

**Why:** The loss already combines sigmoid and BCE stably.

</details>


### Scenario 20: Target out of bounds 🟢 Easy

**Symptoms:** CE reports target 4 out of bounds for 4 logits.

**Code:**

```python
logits.shape==(16,4); y.max()==4
```

**Question:** Which invariant fails?

**Hint:** Valid indices start at zero.

<details><summary>Answer and why</summary>

**Answer:** Ensure `0 <= y < 4` and repair label mapping.

**Why:** Four classes have indices 0–3.

</details>


### Scenario 21: All-zero inputs 🟡 Medium

**Symptoms:** Model cannot learn feature-dependent distinctions.

**Code:**

```python
X=torch.zeros(128,10)
```

**Question:** What data check reveals this?

**Hint:** Inspect mean/std/min/max.

<details><summary>Answer and why</summary>

**Answer:** Repair preprocessing or data ingestion.

**Why:** Identical inputs contain no feature signal.

</details>


### Scenario 22: Suspicious validation score 🟡 Medium

**Symptoms:** Validation accuracy is nearly perfect immediately.

**Code:**

```python
train_dataset=full_dataset; val_dataset=full_dataset
```

**Question:** What methodology bug exists?

**Hint:** Compare sample identities across splits.

<details><summary>Answer and why</summary>

**Answer:** Create disjoint splits and fit preprocessing only on training data.

**Why:** The validation set leaks training examples.

</details>


### Scenario 23: Repeated concatenation 🟡 Medium

**Symptoms:** Output collection becomes slower each batch.

**Code:**

```python
result=torch.cat([result,batch],0)
```

**Question:** Why does cost grow?

**Hint:** Think allocation and copying.

<details><summary>Answer and why</summary>

**Answer:** Append chunks and concatenate once.

**Why:** Each concatenation copies an ever-growing tensor.

</details>


### Scenario 24: Incorrect accuracy dimension 🟡 Medium

**Symptoms:** Accuracy has implausible shape/value.

**Code:**

```python
pred=logits.argmax(dim=0)
```

**Question:** Which dimension represents classes?

**Hint:** Logits are `(batch, classes)`.

<details><summary>Answer and why</summary>

**Answer:** Use `argmax(dim=1)`.

**Why:** Prediction chooses a class for each row.

</details>


### Scenario 25: Model left in eval mode 🟡 Medium

**Symptoms:** Training after first validation learns oddly with Dropout/BN.

**Code:**

```python
for epoch in ...: train_phase(); model.eval(); validate()
```

**Question:** What must begin each training phase?

**Hint:** Modes persist until changed.

<details><summary>Answer and why</summary>

**Answer:** Call `model.train()` before training batches.

**Why:** Validation leaves the module in eval mode.

</details>


## 28. Interview Questions

Practice a 20–40 second short answer first, then reveal the deeper explanation.


### 1. What is your first step when debugging a shape error?

<details><summary>Short interview answer</summary>

Reproduce it on the smallest batch and print shapes at the failing boundary.

</details>

<details><summary>Detailed explanation</summary>

Compare the incoming shape with the operation's documented contract, including batch/channel/class dimensions.

</details>


### 2. Which tensor properties do you inspect first?

<details><summary>Short interview answer</summary>

Shape, dtype, device, and then value finiteness/range.

</details>

<details><summary>Detailed explanation</summary>

Also inspect `requires_grad` and `grad_fn` when gradient flow is relevant.

</details>


### 3. How do you debug a device mismatch?

<details><summary>Short interview answer</summary>

Print the model, input, and target devices and move them consistently.

</details>

<details><summary>Detailed explanation</summary>

Use `next(model.parameters()).device`; move the model once and batches per iteration.

</details>


### 4. How do you debug a dtype mismatch?

<details><summary>Short interview answer</summary>

Print input, target, and parameter dtypes, then compare with the operation contract.

</details>

<details><summary>Detailed explanation</summary>

Embedding/CE indices are commonly long; floating features usually match model precision.

</details>


### 5. What if loss is not decreasing?

<details><summary>Short interview answer</summary>

Validate data, shapes, loss/targets, gradients, updates, LR, mode, then overfit a tiny set.

</details>

<details><summary>Detailed explanation</summary>

Follow a fixed order so each check narrows the hypothesis space.

</details>


### 6. Why overfit a tiny dataset?

<details><summary>Short interview answer</summary>

It tests whether the pipeline can memorize a simple finite sample.

</details>

<details><summary>Detailed explanation</summary>

Failure suggests mechanics, targets, optimization, or capacity before generalization concerns.

</details>


### 7. How verify parameters update?

<details><summary>Short interview answer</summary>

Clone them before one step and compare after `optimizer.step()`.

</details>

<details><summary>Detailed explanation</summary>

Also quantify update magnitude when exact inequality is too coarse.

</details>


### 8. How check whether a parameter got a gradient?

<details><summary>Short interview answer</summary>

Inspect `param.grad` after backward.

</details>

<details><summary>Detailed explanation</summary>

If present, inspect norm and finiteness; if absent, trace graph/frozen state.

</details>


### 9. What does `grad is None` mean?

<details><summary>Short interview answer</summary>

No gradient was populated for that parameter in this backward pass.

</details>

<details><summary>Detailed explanation</summary>

It may be frozen, unused, detached, owned by another model, or backward may be missing.

</details>


### 10. What is a gradient norm?

<details><summary>Short interview answer</summary>

A scalar summary of gradient magnitude.

</details>

<details><summary>Detailed explanation</summary>

Track per-layer/global norms over time; no universal threshold exists.

</details>


### 11. What are vanishing gradients?

<details><summary>Short interview answer</summary>

Gradients become extremely small, especially in earlier layers.

</details>

<details><summary>Detailed explanation</summary>

Updates become ineffective; depth, saturation, and initialization can contribute.

</details>


### 12. What are exploding gradients?

<details><summary>Short interview answer</summary>

Gradient magnitudes grow very large and destabilize updates.

</details>

<details><summary>Detailed explanation</summary>

Symptoms include spikes, huge parameters, and non-finite loss.

</details>


### 13. How does clipping help?

<details><summary>Short interview answer</summary>

It limits gradient norm before the optimizer step.

</details>

<details><summary>Detailed explanation</summary>

It controls update magnitude but does not replace root-cause diagnosis.

</details>


### 14. What commonly causes NaN loss?

<details><summary>Short interview answer</summary>

Bad inputs, invalid math, overflow, high LR, or exploding gradients.

</details>

<details><summary>Detailed explanation</summary>

Find the earliest non-finite tensor, including gradients and parameters.

</details>


### 15. How detect NaN/Inf?

<details><summary>Short interview answer</summary>

Use `isnan`, `isinf`, and `isfinite`.

</details>

<details><summary>Detailed explanation</summary>

Assertions at pipeline boundaries stop corruption close to its source.

</details>


### 16. What does `isfinite` test?

<details><summary>Short interview answer</summary>

Values are neither NaN nor positive/negative infinity.

</details>

<details><summary>Detailed explanation</summary>

It returns an elementwise Boolean tensor; reduce with `.all()` or `.any()`.

</details>


### 17. What is anomaly detection?

<details><summary>Short interview answer</summary>

A slow debugging mode that traces problematic backward operations.

</details>

<details><summary>Detailed explanation</summary>

Scope it around a minimal failing pass and disable it for routine training.

</details>


### 18. When enable anomaly detection?

<details><summary>Short interview answer</summary>

After reproducing a backward/non-finite gradient problem minimally.

</details>

<details><summary>Detailed explanation</summary>

It supplies origin information but carries high runtime overhead.

</details>


### 19. Why can in-place operations break Autograd?

<details><summary>Short interview answer</summary>

They may overwrite values saved for backward.

</details>

<details><summary>Detailed explanation</summary>

Search underscore methods and indexed assignments; version checks detect unsafe mutation.

</details>


### 20. What does an underscore suffix mean?

<details><summary>Short interview answer</summary>

The operation usually mutates the tensor in place.

</details>

<details><summary>Detailed explanation</summary>

Examples include `add_` and `relu_`; mutation is not always wrong, but requires care.

</details>


### 21. How can forgetting eval affect validation?

<details><summary>Short interview answer</summary>

Dropout stays random and BatchNorm uses/updates batch statistics.

</details>

<details><summary>Detailed explanation</summary>

Call eval before validation, then train again before training resumes.

</details>


### 22. How can forgetting no-grad affect validation?

<details><summary>Short interview answer</summary>

Autograd builds graphs, wasting memory and compute.

</details>

<details><summary>Detailed explanation</summary>

Eval changes module behavior; inference/no-grad changes gradient tracking.

</details>


### 23. How can saving loss tensors leak memory?

<details><summary>Short interview answer</summary>

The saved loss may keep its computation graph alive.

</details>

<details><summary>Detailed explanation</summary>

Save `loss.item()` for scalars or a detached CPU tensor when needed.

</details>


### 24. Why does retain_graph increase memory?

<details><summary>Short interview answer</summary>

It keeps graph buffers for another backward.

</details>

<details><summary>Detailed explanation</summary>

Use it only when multiple backward passes through the same graph are intended.

</details>


### 25. What check after CUDA OOM?

<details><summary>Short interview answer</summary>

Sizes, retained outputs/graphs, live models, and CUDA memory stats.

</details>

<details><summary>Detailed explanation</summary>

Then reduce batch size or use AMP/accumulation as appropriate.

</details>


### 26. Allocated vs reserved CUDA memory?

<details><summary>Short interview answer</summary>

Allocated is live tensor memory; reserved includes allocator-held blocks.

</details>

<details><summary>Detailed explanation</summary>

Reserved can exceed allocated due to caching and fragmentation.

</details>


### 27. Does empty_cache fix leaks?

<details><summary>Short interview answer</summary>

No; it cannot free live referenced tensors.

</details>

<details><summary>Detailed explanation</summary>

It only releases unused cached blocks and may not increase usable PyTorch memory.

</details>


### 28. What causes low GPU utilization?

<details><summary>Short interview answer</summary>

Input stalls, small work, host overhead, transfers, or many tiny ops.

</details>

<details><summary>Detailed explanation</summary>

Measure input and compute separately, then profile.

</details>


### 29. How does a slow loader affect GPU?

<details><summary>Short interview answer</summary>

The GPU idles waiting for the next batch.

</details>

<details><summary>Detailed explanation</summary>

Time loader iteration and inspect gaps; benchmark worker/preprocessing choices.

</details>


### 30. How separate loading and compute time?

<details><summary>Short interview answer</summary>

Time loader-only iteration, then time compute on in-memory batches.

</details>

<details><summary>Detailed explanation</summary>

CUDA timing requires synchronization around measured regions.

</details>


### 31. Why are Python loops slow?

<details><summary>Short interview answer</summary>

They add interpreter and often kernel-launch overhead.

</details>

<details><summary>Detailed explanation</summary>

Vectorized operations use larger optimized kernels and better accelerator occupancy.

</details>


### 32. Why vectorize?

<details><summary>Short interview answer</summary>

It expresses bulk work in optimized tensor kernels.

</details>

<details><summary>Detailed explanation</summary>

It improves speed and often clarity, while reducing device dispatch count.

</details>


### 33. Why is repeated cat inefficient?

<details><summary>Short interview answer</summary>

Each step reallocates and copies growing data.

</details>

<details><summary>Detailed explanation</summary>

Collect chunks in a list and concatenate once.

</details>


### 34. Why can item affect GPU performance?

<details><summary>Short interview answer</summary>

It may synchronize device work with the CPU.

</details>

<details><summary>Detailed explanation</summary>

Use it at deliberate logging intervals, not every hot-loop operation.

</details>


### 35. Why synchronize CUDA timing?

<details><summary>Short interview answer</summary>

CUDA launches are asynchronous.

</details>

<details><summary>Detailed explanation</summary>

Without synchronization, a CPU timer may measure dispatch rather than completion.

</details>


### 36. What is warmup?

<details><summary>Short interview answer</summary>

Unmeasured runs before timing.

</details>

<details><summary>Detailed explanation</summary>

It reduces one-time initialization/cache effects in the measured average.

</details>


### 37. What does torch.profiler identify?

<details><summary>Short interview answer</summary>

Where operator time and memory are spent.

</details>

<details><summary>Detailed explanation</summary>

Look for expensive ops, tiny repeated ops, copies, and unexpected CPU work.

</details>


### 38. What is the tiny-batch test?

<details><summary>Short interview answer</summary>

Train on 8–32 samples until the model memorizes them.

</details>

<details><summary>Detailed explanation</summary>

It is a mechanical debugging test, not a generalization metric.

</details>


### 39. How diagnose wrong loss?

<details><summary>Short interview answer</summary>

Match task semantics, output shape, target format/dtype, and raw-logit expectations.

</details>

<details><summary>Detailed explanation</summary>

Single-label multiclass and multilabel tasks have different contracts.

</details>


### 40. How verify multiclass targets?

<details><summary>Short interview answer</summary>

Check long dtype, `(batch,)` shape, and `0 <= y < classes`.

</details>

<details><summary>Detailed explanation</summary>

Also inspect class counts and label-to-index mapping.

</details>


### 41. What is label leakage?

<details><summary>Short interview answer</summary>

Information unavailable at inference leaks target signal into training/evaluation.

</details>

<details><summary>Detailed explanation</summary>

Examples include target-as-feature, overlapping splits, or preprocessing fit on all data.

</details>


### 42. Overfitting vs broken loop?

<details><summary>Short interview answer</summary>

Overfitting has strong train improvement but worsening validation; a broken loop often has flat train loss/updates.

</details>

<details><summary>Detailed explanation</summary>

Verify mechanics and tiny-set memorization before changing regularization.

</details>


### 43. Symptoms of too-high LR?

<details><summary>Short interview answer</summary>

Oscillation, divergence, huge updates, or non-finite values.

</details>

<details><summary>Detailed explanation</summary>

Compare controlled runs and inspect gradient/parameter norms; symptoms are not exclusive.

</details>


### 44. Symptoms of too-low LR?

<details><summary>Short interview answer</summary>

Finite tiny updates and extremely slow loss decrease.

</details>

<details><summary>Detailed explanation</summary>

Confirm gradients and update magnitude before concluding.

</details>


### 45. How debug memory growth each iteration?

<details><summary>Short interview answer</summary>

Inspect stored graph tensors, `retain_graph`, references, and allocated memory over steps.

</details>

<details><summary>Detailed explanation</summary>

Detach/move only needed results and verify the memory curve stabilizes.

</details>


## 29. Knowledge Check Quiz

Choose one answer for each question before opening the separate key.


**1. Loss is constant and parameters never change. First check?**

A. Change architecture
B. Gradients and optimizer step
C. More workers
D. Enable Dropout


**2. `torch.isfinite(x)` tests whether…**

A. x is on GPU
B. values are neither NaN nor ±Inf
C. x requires grad
D. x is contiguous


**3. Why can `losses.append(loss)` grow memory?**

A. Changes batch size
B. Keeps computation graphs alive
C. Disables gradients
D. Changes dtype


**4. Typical class-index CE target dtype?**

A. float64
B. bool
C. long
D. complex64


**5. Conv2d default batch layout?**

A. NHWC
B. NCHW
C. HWNC
D. CHWN


**6. What enters CrossEntropyLoss?**

A. Softmax probabilities
B. Raw logits
C. Argmax labels
D. One scalar


**7. What enters BCEWithLogitsLoss?**

A. Sigmoid probabilities
B. Raw logits
C. Integer class index only
D. Softmax output


**8. `param.grad is None` means…**

A. Gradient is exactly zero
B. No gradient was populated
C. Parameter is NaN
D. Step already happened


**9. When clip gradients?**

A. After backward, before step
B. Before forward
C. After step
D. During eval


**10. What does `model.eval()` do?**

A. Disables all gradients
B. Changes Dropout/BN behavior
C. Moves to CPU
D. Freezes parameters


**11. Best validation context for inference?**

A. enable_grad
B. retain_graph
C. inference_mode
D. detect_anomaly


**12. Which retains a graph intentionally?**

A. item
B. detach
C. retain_graph=True
D. cpu


**13. Allocated CUDA memory is…**

A. Live tensor memory
B. All system RAM
C. Only model weights
D. Disk cache


**14. Does empty_cache free live tensors?**

A. Always
B. Never; references must be released
C. Only gradients
D. Only CPU tensors


**15. Correct CUDA timing needs…**

A. More prints
B. synchronize at boundaries
C. item in loop
D. eval only


**16. Why warm up?**

A. Increase LR
B. Reduce one-time timing effects
C. Allocate huge tensors
D. Reset gradients


**17. Loader-only timing helps isolate…**

A. Initialization
B. Input pipeline
C. Loss choice
D. Gradient clipping


**18. Repeated cat is slow because…**

A. It disables CUDA
B. It repeatedly reallocates/copies
C. It casts dtype
D. It freezes output


**19. Best collection pattern?**

A. cat each batch
B. append chunks, cat once
C. print tensors
D. retain graphs


**20. Too-high LR may cause…**

A. Oscillation/divergence
B. Guaranteed better accuracy
C. No parameters
D. Faster loader


**21. Tiny-set overfit failure suggests…**

A. Perfect generalization
B. Mechanical/optimization issue
C. Loader needs shuffling only
D. Validation leakage


**22. Valid 4-class labels are…**

A. 1–4
B. 0–3
C. 0–4
D. any float


**23. Backbone `.detach()` causes…**

A. More gradients
B. No upstream gradient flow
C. CUDA transfer
D. BatchNorm update


**24. Anomaly detection should be…**

A. Always enabled
B. Scoped to debugging
C. Used only for inference
D. A performance optimization


**25. Underscore methods usually…**

A. Move devices
B. Operate in place
C. Create gradients
D. Profile


**26. Low GPU use with gaps before batches suggests…**

A. Input bottleneck
B. Exploding gradients
C. Wrong loss
D. Label leakage


**27. Frequent `.item()` may…**

A. Synchronize CPU/GPU
B. Vectorize
C. Increase batch size
D. Freeze layers


**28. Suspicious instant validation accuracy may mean…**

A. OOM
B. Leakage
C. Vanishing gradients
D. Wrong device


**29. For logits `(B,C)`, predictions use…**

A. argmax(0)
B. argmax(1)
C. sum(1)
D. mean(0)


**30. First three fast checks are…**

A. LR, epochs, layers
B. shape, dtype, device
C. RAM, disk, network
D. seed, shuffle, print


## Quiz Answer Key

1. **B** · 2. **B** · 3. **B** · 4. **C** · 5. **B** · 6. **B** · 7. **B** · 8. **B** · 9. **A** · 10. **B** · 11. **C** · 12. **C** · 13. **A** · 14. **B** · 15. **B** · 16. **B** · 17. **B** · 18. **B** · 19. **B** · 20. **A** · 21. **B** · 22. **B** · 23. **B** · 24. **B** · 25. **B** · 26. **A** · 27. **A** · 28. **B** · 29. **B** · 30. **B**


## 30. Write From Memory


### Prompt 1

Print a tensor's shape, dtype, and device.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 2

Check whether every value in `x` is finite.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 3

Check whether `x` contains any NaN.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 4

Print every model parameter's gradient norm or `NO GRAD`.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 5

Enable anomaly detection in a scoped context.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 6

Clip global gradient norm to 1.0.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 7

Print the model parameter device.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 8

Snapshot parameters before a step and test which changed.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 9

Measure a CPU operation with `perf_counter`.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 10

Synchronize CUDA before and after timing.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 11

Print allocated CUDA memory only when CUDA exists.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 12

Convert a prediction to detached CPU form.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 13

Replace an elementwise doubling loop with vectorized PyTorch.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 14

Create a tiny-dataset overfit test.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 15

Assert all populated gradients are finite.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 16

Check multiclass label dtype and min/max.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 17

Switch from training to deterministic validation correctly.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 18

Collect batches and concatenate only once.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 19

Calculate a global gradient norm.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


### Prompt 20

Time DataLoader iteration separately from model compute.

<details><summary>Validation idea</summary>

Write a minimal runnable example and use an `assert` or printed invariant to prove it works. Compare with the earlier helper/example only after attempting it.

</details>


## Debugging Flashcards


**Card 1 — Symptom:** Expected all tensors to be on the same device

**First check?**

<details><summary>Answer</summary>

Model device, input device, target device

</details>


**Card 2 — Symptom:** CE complains about target dtype

**First check?**

<details><summary>Answer</summary>

For class indices, usually `torch.long`

</details>


**Card 3 — Symptom:** GPU memory grows every step

**First check?**

<details><summary>Answer</summary>

Stored graph tensors, `retain_graph`, large GPU lists

</details>


**Card 4 — Symptom:** Validation predictions change each call

**First check?**

<details><summary>Answer</summary>

Check `model.eval()` and stochastic preprocessing

</details>


**Card 5 — Symptom:** Parameter gradient is `None`

**First check?**

<details><summary>Answer</summary>

Frozen/unused/detached path or missing backward

</details>


**Card 6 — Symptom:** Gradients grow batch after batch

**First check?**

<details><summary>Answer</summary>

Check missing `zero_grad()`

</details>


**Card 7 — Symptom:** Optimizer steps but model does not change

**First check?**

<details><summary>Answer</summary>

Check gradients, ownership, frozen flags, ordering

</details>


**Card 8 — Symptom:** Loss turns NaN

**First check?**

<details><summary>Answer</summary>

Find first non-finite input/loss/gradient/parameter

</details>


**Card 9 — Symptom:** CUDA timing seems impossibly fast

**First check?**

<details><summary>Answer</summary>

Synchronize around the measured region

</details>


**Card 10 — Symptom:** GPU has long idle gaps

**First check?**

<details><summary>Answer</summary>

Time the input pipeline separately

</details>


**Card 11 — Symptom:** CNN treats width as channels

**First check?**

<details><summary>Answer</summary>

Convert NHWC to NCHW

</details>


**Card 12 — Symptom:** BCE reports target size mismatch

**First check?**

<details><summary>Answer</summary>

Make logits and float targets the same shape

</details>


**Card 13 — Symptom:** Tiny-set accuracy stays at chance

**First check?**

<details><summary>Answer</summary>

Verify loss, gradients, updates, LR, targets, capacity

</details>


**Card 14 — Symptom:** Validation uses much more memory

**First check?**

<details><summary>Answer</summary>

Use inference mode and stop retaining outputs/graphs

</details>


**Card 15 — Symptom:** Output collection slows over time

**First check?**

<details><summary>Answer</summary>

Avoid repeated cat; list then cat once

</details>


## 31. Final Debugging Challenge 🔴 Hard

The following pipeline contains **10 intentional bugs**. It is shown as text so the notebook still runs top-to-bottom. Diagnose it before opening the fixed implementation.

```python
X = torch.randn(600, 20)
y = ((X[:, 0] > 0).long() + (X[:, 1] > 0).long())
y = y.float()                                      # bug
model = Sequential(Linear(20,64), ReLU(), Dropout(.2), Linear(64,4)) # bug
wrong_model = deepcopy(model)
optimizer = Adam(wrong_model.parameters(), lr=10)  # two bugs
model.eval()                                       # bug

for Xb, yb in train_loader:
    logits = model(Xb)                             # device bug on CUDA
    loss = CrossEntropyLoss()(logits, yb)
    loss.backward()
    optimizer.zero_grad()                          # bug: erases grads
    optimizer.step()
    history.append(loss)                           # bug: graph retention

for Xb, yb in val_loader:                          # no eval call and no no-grad
    logits = model(Xb)
    correct += (logits.argmax(dim=0) == yb).sum()  # bug
```

### Required workflow

1. Print `X/y` shapes, dtype, model output shape, devices, and target range.
2. Run one batch; prove loss and gradients are finite and parameters change.
3. Overfit 32 examples to very high training accuracy.
4. Restore full training/validation.
5. Confirm decreasing train loss, deterministic eval, and no graph-connected history.


### Fixed final challenge implementation

This uses three classes (0, 1, 2), one optimizer attached to the forward model, a conservative learning rate, correct ordering, explicit modes, inference context, scalar history, correct accuracy dimension, and consistent devices.


In [ ]:
torch.manual_seed(123)
challenge_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
challenge_X = torch.randn(600, 20)
challenge_y = ((challenge_X[:, 0] > 0).long() + (challenge_X[:, 1] > 0).long())
train_X, val_X = challenge_X[:480], challenge_X[480:]
train_y, val_y = challenge_y[:480], challenge_y[480:]
train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=48, shuffle=True, num_workers=0)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=64, num_workers=0)

challenge_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3)
).to(challenge_device)
challenge_opt = optim.Adam(challenge_model.parameters(), lr=0.01)
challenge_loss_fn = nn.CrossEntropyLoss()

first_X, first_y = next(iter(train_loader))
print("X/y:", first_X.shape, first_y.shape, first_y.dtype)
print("model/X device:", next(challenge_model.parameters()).device, first_X.device)
print("target min/max:", first_y.min().item(), first_y.max().item())
with torch.inference_mode():
    print("output shape:", challenge_model(first_X.to(challenge_device)).shape)
assert first_y.dtype == torch.long and first_y.min() >= 0 and first_y.max() < 3


In [ ]:
challenge_model.train()
one_X, one_y = first_X.to(challenge_device), first_y.to(challenge_device)
one_before = {n: p.detach().clone() for n, p in challenge_model.named_parameters()}
challenge_opt.zero_grad()
one_loss = challenge_loss_fn(challenge_model(one_X), one_y)
assert_finite_tensor("one_loss", one_loss)
one_loss.backward()
assert_gradients_finite(challenge_model)
assert any(p.grad is not None for p in challenge_model.parameters())
challenge_opt.step()
assert_parameters_finite(challenge_model)
assert any(not torch.equal(one_before[n], p.detach()) for n, p in challenge_model.named_parameters())
print("one-batch mechanics verified")


In [ ]:
# Independent 32-sample memorization test
overfit_X = train_X[:32].to(challenge_device)
overfit_y = train_y[:32].to(challenge_device)
overfit_model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3)).to(challenge_device)
overfit_opt = optim.Adam(overfit_model.parameters(), lr=0.03)
for _ in range(180):
    overfit_model.train()
    overfit_opt.zero_grad()
    overfit_loss = challenge_loss_fn(overfit_model(overfit_X), overfit_y)
    overfit_loss.backward()
    overfit_opt.step()
with torch.inference_mode():
    overfit_acc = (overfit_model(overfit_X).argmax(1) == overfit_y).float().mean().item()
print("32-sample accuracy:", overfit_acc)
assert overfit_acc >= 0.95


In [ ]:
train_history = []
for epoch in range(6):
    challenge_model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(challenge_device), batch_y.to(challenge_device)
        challenge_opt.zero_grad()
        batch_loss = challenge_loss_fn(challenge_model(batch_X), batch_y)
        assert_finite_tensor("batch_loss", batch_loss)
        batch_loss.backward()
        assert_gradients_finite(challenge_model)
        challenge_opt.step()
        epoch_loss += batch_loss.item() * len(batch_X)
    train_history.append(epoch_loss / len(train_loader.dataset))

challenge_model.eval()
correct = total = 0
with torch.inference_mode():
    for batch_X, batch_y in val_loader:
        batch_X, batch_y = batch_X.to(challenge_device), batch_y.to(challenge_device)
        predictions = challenge_model(batch_X).argmax(dim=1)
        correct += (predictions == batch_y).sum().item()
        total += len(batch_y)
    probe = val_X[:8].to(challenge_device)
    deterministic_a = challenge_model(probe)
    deterministic_b = challenge_model(probe)

print("train history:", [round(v, 4) for v in train_history])
print("validation accuracy:", correct / total)
assert min(train_history[-2:]) < train_history[0]
assert torch.equal(deterministic_a, deterministic_b)
assert all(isinstance(value, float) for value in train_history)


## 32. Debugging Cheat Sheet 🔥 Memorize This Workflow

```python
# Tensor/model state
print(x.shape, x.dtype, x.device, x.requires_grad)
print(next(model.parameters()).device)

# Values and labels
torch.isnan(x).any(); torch.isinf(x).any(); torch.isfinite(x).all()
print(y.dtype, y.min(), y.max())

# Gradients
for name, param in model.named_parameters():
    print(name, "NO GRAD" if param.grad is None else param.grad.norm().item())
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Debug backward temporarily
with torch.autograd.detect_anomaly():
    loss.backward()

# Validation
model.eval()
with torch.inference_mode():
    predictions = model(X)

# CUDA memory/timing
if torch.cuda.is_available():
    print(torch.cuda.memory_allocated(), torch.cuda.memory_reserved())
    torch.cuda.synchronize(); start = time.perf_counter()
    output = model(X)
    torch.cuda.synchronize(); elapsed = time.perf_counter() - start
```

### Training is broken?

1. Can one batch run?
2. Are shapes, dtypes, and devices correct?
3. Are targets valid and loss inputs correct?
4. Is loss finite?
5. Does backward create finite gradients?
6. Does `step()` change parameters?
7. Can the model overfit 32 samples?
8. Are train/eval and gradient contexts correct?
9. Is memory growing?
10. Is the DataLoader slow?
11. Only then tune architecture/hyperparameters.

### Performance quick reference

```text
GPU slow? → loader → batch/model size → transfers → Python loops/sync → profile
Memory growing? → stored losses/outputs → retain_graph → live references → memory stats
```


## Before Moving to 10_transformer_pytorch.ipynb

- [ ] I have a systematic PyTorch debugging workflow.
- [ ] I inspect shape, dtype, and device first.
- [ ] I can diagnose Linear/CNN and target shape mismatches.
- [ ] I can diagnose dtype, loss, activation, and target-format problems.
- [ ] I know what to check when loss does not decrease.
- [ ] I can verify parameter updates and inspect gradient flow.
- [ ] I understand `grad=None`, exploding gradients, and vanishing gradients.
- [ ] I can detect and trace NaN/Inf values.
- [ ] I understand anomaly detection and in-place Autograd errors.
- [ ] I can debug train/eval and gradient-context mistakes.
- [ ] I can identify graph-retention leaks and investigate CUDA OOM.
- [ ] I can diagnose input bottlenecks and low GPU utilization.
- [ ] I can time CPU/CUDA work and use the basic profiler.
- [ ] I understand vectorization and hot-loop anti-patterns.
- [ ] I can use a tiny-dataset overfitting test.
- [ ] I can debug a training pipeline methodically.
